In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Load dataset
df = pd.read_csv("diamonds.csv")
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [3]:
# Features and target
X = df.drop("price", axis=1)
y = df["price"]

# Identify columns
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(include="object").columns

In [4]:
# Preprocessing
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OrdinalEncoder(), cat_cols)
    ]
)

# Model
knn = KNeighborsRegressor(n_neighbors=7)

# Pipeline
pipeline = Pipeline([
    ("preprocessing", preprocess),
    ("model", knn)
])

In [5]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Train
pipeline.fit(X_train, y_train)

# Predictions
y_pred = pipeline.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 392.59345304306373
RMSE: 747.1007400018651
R2 Score: 0.9644703714710141


In [6]:
# Sample Prediction
sample = pd.DataFrame({
    "carat":[0.7],
    "cut":["Ideal"],
    "color":["E"],
    "clarity":["SI1"],
    "depth":[61.5],
    "table":[55],
    "x":[5.7],
    "y":[5.6],
    "z":[3.5]
})

prediction = pipeline.predict(sample)

print("Predicted Price:", prediction[0])

Predicted Price: 2904.0


In [7]:
# Save model
with open("diamond_knn_model.pkl", "wb") as f:
    pickle.dump(pipeline, f)

print("Model saved successfully!")

Model saved successfully!


In [8]:
import sklearn
print(sklearn.__version__)

1.6.1
